# 벡터 정규화에서 `axis`와 `keepdims` 추론

- 기준 TIL: [2026-08-13](../../til/2026/08/2026-08-13.md)
- 관련 강의자료: [1장 1강: 벡터의 수학적 정의와 기하학적 해석](../../materials/private/kant-basic-math/01-01_벡터의_정의와_기하학적_해석.pdf)
- 난이도: Core
- 상태: 시작 전


## 왜 지금 이 실습을 하는가

- TIL에서 확인된 이해: NumPy 배열의 `ndim`과 수학적 공간의 차원을 구분했고, L2 정규화는 벡터의 노름을 1로 만드는 연산이며 영벡터는 예외라고 정리했다. 또한 `axis`는 계산하면서 줄이는 축이고 `keepdims=True`는 그 축을 크기 1로 남긴다고 설명했다.
- 이번에 확인할 부족한 부분: TIL에는 실행된 출력이나 새로운 Tensor shape에서 `axis`, 중간 노름 shape, broadcasting 결과를 해석한 근거가 없다.
- 핵심 질문: 여러 벡터가 들어 있는 Tensor에서 어느 축을 줄여야 각 벡터를 개별적으로 L2 정규화할 수 있으며, `keepdims`는 나눗셈 shape에 어떤 영향을 주는가?


## 선수개념과 완료 기준

선수개념은 벡터의 L2 노름, 영벡터의 예외, NumPy의 `shape`와 `axis`이다.

- [ ] 코드를 실행하기 전에 `H`, `norms`, `H_unit`의 shape을 직접 예측했다.
- [ ] 각 축이 무엇을 나타내는지 쓰고, 정규화할 축을 직접 선택했다.
- [ ] 정규화 후 각 벡터의 노름을 확인하고 예상과 관찰을 비교했다.
- [ ] `keepdims=False`일 때 나눗셈에서 관찰한 결과를 broadcasting 규칙으로 설명했다.
- [ ] 영벡터가 포함될 때 생기는 한계를 직접 관찰하고 이유를 설명했다.
- [ ] 결과를 자신의 말로 정리하고 `knowledge/`에 반영할 만한 이해인지 판단했다.


## 실행 전 예상

아래 코드를 실행하기 전에 먼저 답을 적는다.

1. `H.shape == (2, 3, 4)`에서 각 축은 무엇을 나타낸다고 정할 것인가?
2. `H[batch_index, token_index, :]`를 벡터 하나로 본다면 어느 축을 따라 노름을 계산해야 하는가?
3. `keepdims=True`일 때 `norms.shape`은 무엇이라고 예상하는가?
4. `H / norms`의 결과 shape은 무엇이라고 예상하는가?
5. `keepdims=False`로 바꾸면 어떤 일이 생길 것 같은가?

- 내 예상: 
  1. 첫번째 축인 2는 데이터 셋의 개수, 3은 rows 수, 4는 feature수를 나타낸다.
  2. axis 2
  3. (2, 3, 1)
  4. (2, 3, 4)
  5. 차원이 하나 준다. H와 바로 나누면 broadcasting 오류가 발생한다.


In [1]:
# 준비: 이 셀은 실행해도 되지만, 아래 실험 셀은 shape을 먼저 예측한 뒤 실행하세요.
import numpy as np

H = np.arange(1, 25, dtype=float).reshape(2, 3, 4)

print("H.shape:", H.shape)
print(H)


H.shape: (2, 3, 4)
[[[ 1.  2.  3.  4.]
  [ 5.  6.  7.  8.]
  [ 9. 10. 11. 12.]]

 [[13. 14. 15. 16.]
  [17. 18. 19. 20.]
  [21. 22. 23. 24.]]]


## 1. 마지막 축의 벡터를 개별적으로 정규화하기

1. `axis_to_normalize`에 정규화할 축을 지정한다.
2. 실행 전에 `norms`와 `H_unit`의 shape을 주석에 적는다.
3. 실행 결과가 예상과 같은지 확인한다.
4. 마지막 출력의 각 값이 무엇을 뜻하는지 설명한다.

<details>
<summary>힌트 1</summary>

벡터 하나는 `H[batch_index, token_index, :]`처럼 선택된다. 콜론으로 남긴 축이 벡터의 성분 축이다.
</details>

<details>
<summary>힌트 2</summary>

노름을 계산하면 벡터의 성분 축은 하나의 숫자로 줄어야 한다. 줄어들지 않는 앞의 축들은 각각 어떤 벡터의 노름인지 식별하는 역할을 한다.
</details>

<details>
<summary>힌트 3</summary>

NumPy에서 마지막 축은 `-1`로도 지정할 수 있다. `keepdims=True`는 계산한 축의 자리를 크기 1로 남긴다.
</details>


In [9]:
# TODO 1: 정규화할 축을 선택하세요.
axis_to_normalize = 2
assert axis_to_normalize is not None, "실행 전에 정규화할 axis를 지정하세요."

# 예상 norms.shape: 2, 3, 1
norms = np.linalg.norm(H, axis=axis_to_normalize, keepdims=True)

# 예상 H_unit.shape: 2, 3, 4
H_unit = H / norms

print("H.shape:", H.shape)
print("norms.shape:", norms.shape)
print("H_unit.shape:", H_unit.shape)
print("정규화 후 노름:", np.linalg.norm(H_unit, axis=axis_to_normalize))


H.shape: (2, 3, 4)
norms.shape: (2, 3, 1)
H_unit.shape: (2, 3, 4)
정규화 후 노름: [[1. 1. 1.]
 [1. 1. 1.]]


## 2. `keepdims=False`와 영벡터 한계 확인

이 절은 같은 정규화 메커니즘의 shape 조건과 실패 사례를 확인한다.

1. `keepdims=False`로 노름을 계산하기 전에 결과 shape을 예측한다.
2. 그 노름으로 `H`를 바로 나눌 수 있을지 예측한 뒤 실행한다.
3. `H_with_zero[0, 0, :]`를 영벡터로 바꾸고 정규화를 시도한다.
4. 경고나 결과를 그대로 관찰하되, 아직 안전 처리 코드를 완성하지 말고 필요한 조건을 말로 적는다.

<details>
<summary>막혔을 때 힌트</summary>

broadcasting은 뒤쪽 축부터 크기를 비교한다. 두 크기가 같거나 둘 중 하나가 1이어야 한다. 영벡터의 노름은 0이므로 나눗셈의 분모를 확인한다.
</details>


In [6]:
# TODO 2-A: keepdims=False일 때를 관찰하세요.
norms_without_keepdims = np.linalg.norm(
    H, axis=axis_to_normalize, keepdims=False
)
print("norms_without_keepdims.shape:", norms_without_keepdims.shape)

try:
    H_divided = H / norms_without_keepdims
    print("나눗셈 결과 shape:", H_divided.shape)
except ValueError as error:
    print("관찰한 오류:", error)

# TODO 2-B: 영벡터를 포함했을 때를 관찰하세요.
H_with_zero = H.copy()
H_with_zero[0, 0, :] = 0.0

# 아래 두 줄의 TODO를 직접 작성한 뒤 결과와 경고를 관찰하세요.
zero_norms = np.linalg.norm(
    H_with_zero,
    axis=axis_to_normalize,
    keepdims=True,
)

H_with_zero_unit = H_with_zero / zero_norms

print("zero_norms.shape:", zero_norms.shape)
print("영벡터의 노름:", zero_norms[0, 0, :])
print("정규화된 영벡터:", H_with_zero_unit[0, 0, :])
print("NaN 여부:", np.isnan(H_with_zero_unit[0, 0, :]))


norms_without_keepdims.shape: (2, 3)
관찰한 오류: operands could not be broadcast together with shapes (2,3,4) (2,3) 
zero_norms.shape: (2, 3, 1)
영벡터의 노름: [0.]
정규화된 영벡터: [nan nan nan nan]
NaN 여부: [ True  True  True  True]


/tmp/ipykernel_67333/2236881182.py:24: RuntimeWarning: invalid value encountered in divide
  H_with_zero_unit = H_with_zero / zero_norms


## 결과 해석과 마무리

- 실제로 확인한 `H`, `norms`, `H_unit`의 shape: (2, 3, 4), (2, 3, 1), (2, 3, 4)
- 실행 전 예상과 같거나 달랐던 부분: keepdims=True 의 shape와 정규화 결과는 예상과 같았다. keepdims=False 에서도 노름의 차원이 하나 줄고 broadcasting 오류가 발생한다는 예쌍이 맞았다. 영벡터에서는 오류가 발생하기보다 nan과 divide by zero 경고가 나타났다.
- `axis`가 줄이는 축이라는 말이 이번 Tensor에서 구체적으로 뜻하는 것: 이번 예시에서는 axis=2에 있던 벡터 성분 4개를 벡터마다 1개의 노름으로 줄였다. 앞의 (2, 3) 은 어떤 벡터의 노름인지 구분하기 위해 남았다.
- `keepdims=True`가 broadcasting을 가능하게 만든 과정: 노름의 shape을 (2, 3, 1)로 유지했기 때문에 마지막 축의 1이 broadcasting으로 확장되어 각 벡터의 네 성분을 같은 노름으로 나눌 수 있었따.
- 정규화 후 마지막 출력 값들이 보여주는 것: (2, 3)에 들어 있는 여섯 값이 모두 1로 출력됐다. 이는 총 6개의 벡터가 각각 길이 1로 정규화됐다는 뜻이다.
- `keepdims=False`에서 관찰한 결과와 그 이유: 노름이 (2, 3)이 되면서 (2, 3, 4)와 오른쪽부터 비교햇을 때, 4와 3이 맞지 않아 broadcasting이 실패했다.
- 영벡터에서 관찰한 결과와 필요한 안전 조건: 영벡터의 노름은 0이고 이를 분모로 사용하면 0 / 0 이 되어 nan이 발생한다. 따라서 정규화 전에 노름이 0이거나 너무 작은 값인지 확인하거나, 영벡터를 어떻게 처리할지 별도의 규칙이 필요하다.
- 이 실험은 L2 정규화와 특성별 Min-max scaling의 차이에 대해 무엇을 보여주는가? L2정규화는 각 벡터를 자신의 노름으로 나누어 벡터의 크기를 1로 만든다. Min-max scaling은 각 특성의 최솟값과 최댓값을 이용해 범위를 조절한다.
- 이 실험의 한계: 
  - 작은 인공 Tensor만 사용했다
  - 0에 매우 가까운 노름은 확인하지 않았다.
  - 실제 임베딩처럼 값의 차원이 큰 경우는 확인하지 않았다.
